In [ ]:
# Import custom preprocessing functions and utilities
from preprocess import *


In [ ]:
# 1. Load the raw dataset and perform an initial overview of missing values
csv = r"data/pancreatic_cancer_dataset.csv"
df = pd.read_csv(csv)
overview(df).sort_values(by="missing")


In [ ]:
# 2. Standardize column names and drop the unique patient identifier
df = uniform(df).drop("patient_id", axis=1)
df


In [ ]:
# 3. Handle missing values: fill missing categorical/text features with 'unknown'
summary = overview(df)
nan_cols = list(summary[summary["missing"] > 0].index)
for col in nan_cols:
    df[col] = df[col].fillna("unknown")
overview(df).sort_values(by="missing")


In [ ]:
# 4. Check data types, numerical ranges, and unique values for categorical features
for col in df.select_dtypes(include=["number"]).columns:
    print(col, df[col].dtype, "min:", df[col].min(), "max:", df[col].max())
for col in df.select_dtypes(exclude=["number"]).columns:
    print(col, df[col].dtype, "unique:", df[col].unique())


In [ ]:
# 5. Check the minimum and maximum diagnosis dates
df["diagnosis_date"].min(), df["diagnosis_date"].max()


In [ ]:
# 6. Convert diagnosis date into a numerical timestamp format for machine learning models
df["diagnosis_date"] = pd.to_datetime(df["diagnosis_date"]).astype("int64") // 10**9
df


In [ ]:
# 7. Convert the target variable 'survived' into a binary format (1 / 0) and check class balance
df["survived"] = (df["survived"] == "yes").astype(int)
df["survived"].value_counts()


In [ ]:
# 8. Perform stratified train/validation/test split to preserve class distribution
y = df["survived"]
df_full_train, df_test, y_full_train, y_test = train_test_split(
    df, y, test_size=0.2, random_state=1, stratify=y
)
df_train, df_val, y_train, y_val = train_test_split(
    df_full_train, y_full_train, test_size=0.25, random_state=1, stratify=y_full_train
)


In [ ]:
# 9. Plot the correlation matrix of numerical features (computed strictly on the train set)
numeric_df = df_train.select_dtypes(include=["number"])
plt.figure(figsize=(12, 10))
sns.heatmap(numeric_df.corr(), annot=False, cmap="coolwarm", center=0)
plt.title("Correlation Matrix of Numeric Features")
plt.tight_layout()
plt.show()


In [ ]:
# 10. Drop target variable, redundant year, and target-leaking features (Data Leakage prevention)
garbage = ["survived", "diagnosis_year", "five_year_status", "survival_months"]
df_full_train = df_full_train.drop(garbage, axis=1)
df_test = df_test.drop(garbage, axis=1)
df_train = df_train.drop(garbage, axis=1)
df_val = df_val.drop(garbage, axis=1)
df_train.head().T


In [ ]:
# 11. Apply preprocessing pipeline (fitted strictly on train, then transformed on val, full_train, and test)
X_train = preprocessor.fit_transform(df_train)
X_val = preprocessor.transform(df_val)
X_full_train = preprocessor.transform(df_full_train)
X_test = preprocessor.transform(df_test)


In [ ]:
# 12. Hyperparameter tuning for Logistic Regression using Optuna
study = optuna.create_study(
    direction="maximize", pruner=optuna.pruners.MedianPruner(n_warmup_steps=10)
)
study.optimize(
    lambda trial: objective(
        trial,
        X_train,
        y_train,
        X_val,
        y_val,
        "lin_c",
    ),
    n_trials=50,
)


In [ ]:
study.best_value


In [ ]:
lin_params = study.best_params
lin_params


In [ ]:
# Train and evaluate Logistic Regression on the holdout test set
lin_model = LogisticRegression(
    **lin_params, max_iter=10000, random_state=1, solver="saga", penalty="elasticnet"
)
lin_model.fit(X_full_train, y_full_train)
y_pred = lin_model.predict_proba(X_test)
roc_auc_score(y_test, y_pred[:, 1])


In [ ]:
# 13. Hyperparameter tuning for Random Forest using Optuna
study = optuna.create_study(
    direction="maximize", pruner=optuna.pruners.MedianPruner(n_warmup_steps=10)
)
study.optimize(
    lambda trial: objective(
        trial,
        X_train,
        y_train,
        X_val,
        y_val,
        "for_c",
    ),
    n_trials=50,
)


In [ ]:
study.best_value


In [ ]:
for_params = study.best_params
for_params


In [ ]:
# Train and evaluate Random Forest on the holdout test set
for_model = RandomForestClassifier(**for_params, random_state=1)
for_model.fit(X_full_train, y_full_train)
y_pred = for_model.predict_proba(X_test)
roc_auc_score(y_test, y_pred[:, 1])


In [ ]:
# 14. Hyperparameter tuning for Extra Trees using Optuna
study = optuna.create_study(
    direction="maximize", pruner=optuna.pruners.MedianPruner(n_warmup_steps=10)
)
study.optimize(
    lambda trial: objective(
        trial,
        X_train,
        y_train,
        X_val,
        y_val,
        "xtr_c",
    ),
    n_trials=50,
)


In [ ]:
study.best_value


In [ ]:
xtr_params = study.best_params
xtr_params


In [ ]:
# Train and evaluate Extra Trees on the holdout test set
xtr_model = ExtraTreesClassifier(**xtr_params, random_state=1)
xtr_model.fit(X_full_train, y_full_train)
y_pred = xtr_model.predict_proba(X_test)
roc_auc_score(y_test, y_pred[:, 1])


In [ ]:
# 15. Hyperparameter tuning for CatBoost using Optuna
study = optuna.create_study(
    direction="maximize", pruner=optuna.pruners.MedianPruner(n_warmup_steps=10)
)
study.optimize(
    lambda trial: objective(
        trial,
        X_train,
        y_train,
        X_val,
        y_val,
        "cgb_c",
    ),
    n_trials=50,
)


In [ ]:
study.best_value


In [ ]:
cgb_params = study.best_params
cgb_params


In [ ]:
# Train and evaluate CatBoost on the holdout test set
cgb_model = CatBoostClassifier(**cgb_params, random_state=1)
cgb_model.fit(X_full_train, y_full_train)
y_pred = cgb_model.predict_proba(X_test)
roc_auc_score(y_test, y_pred[:, 1])


In [ ]:
# 16. Hyperparameter tuning for LightGBM using Optuna
study = optuna.create_study(
    direction="maximize", pruner=optuna.pruners.MedianPruner(n_warmup_steps=10)
)
study.optimize(
    lambda trial: objective(
        trial,
        X_train,
        y_train,
        X_val,
        y_val,
        "lgb_c",
    ),
    n_trials=50,
)


In [ ]:
study.best_value


In [ ]:
lgb_params = study.best_params
lgb_params


In [ ]:
# Train and evaluate LightGBM on the holdout test set
lgb_model = LGBMClassifier(**lgb_params, random_state=1)
lgb_model.fit(X_full_train, y_full_train)
y_pred = lgb_model.predict_proba(X_test)
roc_auc_score(y_test, y_pred[:, 1])


In [ ]:
# 17. Hyperparameter tuning for XGBoost using Optuna
study = optuna.create_study(
    direction="maximize", pruner=optuna.pruners.MedianPruner(n_warmup_steps=10)
)
study.optimize(
    lambda trial: objective(
        trial,
        X_train,
        y_train,
        X_val,
        y_val,
        "xgb_c",
    ),
    n_trials=50,
)


In [ ]:
study.best_value


In [ ]:
xgb_params = study.best_params
xgb_params


In [ ]:
# Train and evaluate the winning model (XGBoost) on the holdout test set
xgb_model = XGBClassifier(**xgb_params, random_state=1)
xgb_model.fit(X_full_train, y_full_train)
y_pred = xgb_model.predict_proba(X_test)
roc_auc_score(y_test, y_pred[:, 1])


 XGBClassifier is the best model with the highest AUC score of 0.8524.
 Here are the best parameters for XGBClassifier:

 {'n_estimators': 111,
  'learning_rate': 0.01271966900107984,
  'max_depth': 5,
  'subsample': 0.7337999287278185,
  'colsample_bytree': 0.8603499573440633}

In [ ]:
# 18. Save the final pipeline (preprocessor + best model) into a binary file
with open("model.bin", "wb") as f_out:
    pickle.dump((preprocessor, xgb_model), f_out)